# Chapter 4: Machine Learning Model Development

## 4.1 Introduction
This notebook implements the predictive modeling phase.

**Algorithms Implemented:**
1.  **Logistic Regression** (Baseline)
2.  **Random Forest Classifier** (Ensemble)
3.  **XGBoost** (Gradient Boosting)
4.  **Deep Neural Network** (TensorFlow/Keras)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

np.random.seed(42)
tf.random.set_seed(42)

## 4.2 Data Loading
We attempt to load the **REAL** preprocessed data from Notebook 02. If that data is not found (preprocessing not run yet), we fall back to **SYNTHETIC** data for code verification.

In [ ]:
def load_real_data(data_dir='../data/processed'):
    """
    Loads processed genomic and transcriptomic data.
    Expected files: 'genotype_matrix.csv', 'expression_matrix.csv', 'targets.csv'
    """
    try:
        genotypes = pd.read_csv(os.path.join(data_dir, 'genotype_matrix.csv'), index_col=0)
        expression = pd.read_csv(os.path.join(data_dir, 'expression_matrix.csv'), index_col=0)
        targets = pd.read_csv(os.path.join(data_dir, 'targets.csv'), index_col=0)
        
        # Merge on Sample ID
        X = pd.concat([genotypes, expression], axis=1, join='inner')
        y = targets.loc[X.index]
        
        print(f"Successfully loaded REAL data. Shape: {X.shape}")
        return X.values, y.values.ravel()
    except FileNotFoundError:
        print("Real processed data not found. Falling back to synthetic generator.")
        return None, None

def generate_synthetic_data(n_samples=1000, n_snps=450, n_expr=50):
    print("Generating SYNTHETIC data for testing model architecture...")
    maf = 0.2
    X_snps = np.random.binomial(2, maf, size=(n_samples, n_snps))
    X_expr = np.random.normal(loc=5, scale=2, size=(n_samples, n_expr))
    X = np.hstack((X_snps, X_expr))
    
    risk_score = (
        0.5 * X_snps[:, 0] + 0.5 * X_snps[:, 10] + 0.3 * X_snps[:, 20] + 
        -0.2 * X_expr[:, 0] + np.random.normal(0, 0.5, size=n_samples)
    )
    threshold = np.percentile(risk_score, 90)
    y = (risk_score > threshold).astype(int)
    return X, y

# EXECUTION LOGIC
X, y = load_real_data()
if X is None:
    X, y = generate_synthetic_data()

## 4.3 Data Splitting & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

## 4.4 Model 1: Random Forest (Baseline)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

## 4.5 Model 2: XGBoost (Advanced Gradient Boosting)

In [ ]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

## 4.6 Model 3: Deep Neural Network (Deep Learning)

In [ ]:
model_dnn = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_dnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])

history = model_dnn.fit(
    X_train_scaled, y_train, 
    epochs=50, 
    batch_size=32, 
    validation_split=0.1, 
    verbose=0 
)

loss, acc, auc = model_dnn.evaluate(X_test_scaled, y_test)
print(f"\nDNN Accuracy: {acc:.4f}, AUC: {auc:.4f}")

## 4.7 Model Comparison (ROC Curves)

In [ ]:
plt.figure(figsize=(10, 6))

# Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_model.predict_proba(X_test)[:,1])
auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.2f})')

# XGBoost
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_model.predict_proba(X_test)[:,1])
auc_xgb = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1])
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {auc_xgb:.2f})')

# DNN
y_pred_dnn = model_dnn.predict(X_test_scaled).ravel()
fpr_dnn, tpr_dnn, _ = roc_curve(y_test, y_pred_dnn)
auc_dnn = roc_auc_score(y_test, y_pred_dnn)
plt.plot(fpr_dnn, tpr_dnn, label=f'Neural Network (AUC = {auc_dnn:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Neural Tube Defect Prediction')
plt.legend()
plt.show()